# Task 2: Document Q&A Assistant with RAG & Vector Embeddings

**Project:** DocQA, document question answering with citations and refusal
**Language and stack:** Python 3.12, Retrieval-Augmented Generation (RAG), Chroma vector store, OpenRouter (`openai/text-embedding-3-small`, `openai/gpt-oss-20b`), LangChain text splitter, pytest, ruff, uv

This notebook walks through my Task 2 project from top to bottom. It does not copy the code. It imports the `docqa` package, shows the source of each module, and runs the real pipeline, so what you read here is the same code the tests check.

## Contents

1. Objective and key concepts
2. Setup
3. Pipeline architecture
4. Source code tour (config, loader, chunking, embeddings, store, retriever, answer, pipeline, conversation, telemetry, evaluation, cli)
5. The grounding prompt
6. The document corpus
7. Live run: ingest the documents
8. Live run: ask a question and read the citations
9. Live run: a question the documents cannot answer
10. Conversation memory
11. Telemetry
12. Evaluation results
13. Tests
14. Design decisions, limitations, skills

## 1. Objective and key concepts

**Objective:** build a document question-answering system that retrieves the relevant passages from a set of documents and writes answers **only** from them, with numbered source citations. When the documents do not contain the answer, the assistant says so instead of guessing.

**Key concepts**

- Retrieval-Augmented Generation (RAG)
- Document processing: Markdown, plain text and PDF loading, with page numbers kept for PDFs
- Text chunking: heading-aware, 800 characters with 100 overlap, line spans kept
- Vector embeddings (`openai/text-embedding-3-small` through OpenRouter)
- Semantic search: Chroma vector store with cosine distance, top-k retrieval and a relevance threshold
- Grounded prompting, numbered citations and a fixed refusal sentence
- Idempotent ingestion: unchanged files are skipped
- Conversation memory with follow-up question rewriting
- Retrieval and answer evaluation, plus telemetry (latency and tokens)

## 2. Setup

In [5]:
import json
import os
import subprocess
import sys
from pathlib import Path

SKIP_FOLDERS = {".venv", "venv", "node_modules", ".git", "__pycache__", ".ipynb_checkpoints"}


def find_project_root() -> Path:
    """Find the folder that contains the docqa package.

    Works when the notebook sits in the repository root, inside the DocQA
    project folder, or in any folder in between.
    """
    here = Path.cwd().resolve()

    # 1. This folder or one of its parents (notebook opened inside the project).
    for folder in [here, *here.parents]:
        if (folder / "docqa" / "cli.py").exists():
            return folder

    # 2. A project folder below this one (notebook opened from the repository root).
    for current, subfolders, _ in os.walk(here):
        depth = len(Path(current).relative_to(here).parts)
        subfolders[:] = (
            [d for d in subfolders if d not in SKIP_FOLDERS and not d.startswith(".")]
            if depth < 3 else []
        )
        if (Path(current) / "docqa" / "cli.py").exists():
            return Path(current)

    raise FileNotFoundError(
        "Could not find the docqa package (a folder named 'docqa' containing cli.py). "
        "Start Jupyter from the repository root or from the DocQA project folder."
    )


STARTED_IN = Path.cwd().resolve()
PROJECT = find_project_root()
os.chdir(PROJECT)  # data, evaluation and .env paths below are relative to the project folder
if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))

print("Notebook started in:", STARTED_IN.name)
print("Project folder found:", PROJECT.name)
print("Python:", sys.version.split()[0])

Notebook started in: Task2_DocQA_RAG
Project folder found: Task2_DocQA_RAG
Python: 3.12.12


In [6]:
# Check that the API key is configured, without printing any secret.
env_file = PROJECT / ".env"
print(".env file found:", env_file.exists())
if env_file.exists():
    names = set()
    for line in env_file.read_text(encoding="utf-8").splitlines():
        line = line.strip()
        if "=" in line and not line.startswith("#"):
            names.add(line.split("=", 1)[0].strip())
    print("OPENROUTER_API_KEY present:", "OPENROUTER_API_KEY" in names)

.env file found: False


This notebook can live in the repository root or inside the DocQA project folder. The first cell finds the project folder by itself and switches into it, so the data, evaluation and `.env` paths all resolve correctly. Start Jupyter from the repository root with `uv run --with jupyter jupyter lab`.

## 3. Pipeline architecture

```
 documents (.md .txt .pdf)
        |  loader: NFKC-normalised text, page numbers for PDFs
        v
 chunking: heading-aware, 800 chars / 100 overlap, line spans kept
        |
        v  embeddings (OpenRouter, openai/text-embedding-3-small)
 Chroma vector store (cosine) -- idempotent: unchanged files are skipped
        ^
        |  question -> embed -> top-k chunks -> relevance threshold
        |                                   |
     ingest                                 v
                       grounded prompt -> LLM (openai/gpt-oss-20b)
                                   |
                                   v
                    answer + numbered citations, or the fixed refusal
```

Two layers protect against invented answers:

1. If no retrieved chunk passes the relevance threshold, the refusal is returned **without calling the LLM**.
2. Otherwise the prompt tells the model to answer only from the numbered context and to reply with the exact refusal sentence when the answer is absent. A refusal clears the citations.

**Citations.** PDFs cite file and page. Markdown and text cite file, section and line span, for example `about.md | About Insurellm, lines 1-5`.

## 4. Source code tour

The helper below prints the source of a module straight from the package, so this notebook always matches the real code.

In [7]:
import importlib
import inspect


def show_source(module_name: str) -> None:
    try:
        module = importlib.import_module(module_name)
        source = inspect.getsource(module)
    except Exception as error:  # keep "Run All Cells" going if one module cannot be shown
        print(f"Could not show {module_name}: {error}")
        return
    print(f"# {module_name}  ({len(source.splitlines())} lines)\n")
    print(source)

### 4.1 Configuration (`config.py`)

Validated settings from `.env` or the environment: models, chunk size, `top_k`, relevance threshold, retry policy and paths. Invalid values stop the program with a clear error.

In [8]:
show_source("docqa.config")

# docqa.config  (132 lines)

"""Runtime configuration for the DocQA pipeline.

Settings are built from environment variables. The environment is injected as
a mapping, so tests pass a plain dict and never touch the real environment.
Everything is validated once, up front, so a bad value fails fast with a
readable message instead of crashing deep inside the pipeline.
"""

import os
from collections.abc import Mapping
from pathlib import Path
from typing import Literal, Self

from pydantic import BaseModel, ConfigDict, Field, ValidationError, model_validator

from docqa.errors import ConfigError

# Project root = the folder that contains the `docqa` package (no absolute paths).
PROJECT_ROOT = Path(__file__).resolve().parents[1]

DEFAULT_LOG_LEVEL = "INFO"
DEFAULT_CHUNK_SIZE = 800
DEFAULT_CHUNK_OVERLAP = 100
DEFAULT_TOP_K = 10
# Tuned on the InsureLM eval: weakest answerable top hit 0.31, strongest unrelated 0.17.
DEFAULT_RELEVANCE_THRESHOLD = 0.25
DEFAULT_MAX_QUESTION_CHARS = 500
DEFAULT

### 4.2 Document loading (`loader.py`)

Reads `.md`, `.txt` and `.pdf` files, normalises the text (NFKC) and keeps page numbers for PDFs.

In [9]:
show_source("docqa.loader")

# docqa.loader  (130 lines)

"""Read .pdf, .md and .txt files into text documents that remember their origin."""

import logging
import unicodedata
from dataclasses import dataclass, field
from pathlib import Path

import pymupdf

from docqa.errors import (
    DocQAError,
    DocumentReadError,
    EmptyDocumentError,
    InputTooLargeError,
    UnsupportedFileError,
)

logger = logging.getLogger(__name__)

PDF_SUFFIX = ".pdf"
TEXT_SUFFIXES = frozenset({".md", ".txt"})
SUPPORTED_SUFFIXES = TEXT_SUFFIXES | {PDF_SUFFIX}
DEFAULT_MAX_BYTES = 20 * 1024 * 1024


@dataclass(frozen=True)
class LoadedDocument:
    """Text from one text file, or from one page of a PDF."""

    source: Path
    text: str
    page: int | None = None  # 1-indexed for PDFs, None for text files


@dataclass
class LoadReport:
    """Outcome of loading a directory: the documents read and the files skipped."""

    documents: list[LoadedDocument] = field(default_factory=list)
    failed: dict[Path, str] = field(default

### 4.3 Chunking (`chunking.py`)

Heading-aware splitting with line spans, so a citation can point to a section and exact lines. LangChain's `RecursiveCharacterTextSplitter` does the splitting.

In [10]:
show_source("docqa.chunking")

# docqa.chunking  (211 lines)

"""Split loaded documents into chunks that carry citation metadata.

Markdown files are split on headings first, so every chunk knows its section and
line span. Long sections are then split by LangChain's RecursiveCharacterTextSplitter.
PDFs are split page by page, so every chunk keeps its page number.
"""

import logging
import re
from collections.abc import Sequence
from dataclasses import dataclass
from pathlib import Path

from langchain_text_splitters import RecursiveCharacterTextSplitter

from docqa.errors import ConfigError
from docqa.loader import LoadedDocument

logger = logging.getLogger(__name__)

MARKDOWN_SUFFIXES = frozenset({".md", ".markdown"})
HEADING_RE = re.compile(r"^(#{1,6})\s+(.*\S)\s*$")
FENCE_RE = re.compile(r"^\s*(`{3}|~{3})")
BREADCRUMB_SEP = " > "


@dataclass(frozen=True)
class Section:
    """A run of text under one heading; lines are 1-indexed and inclusive."""

    path: tuple[str, ...]
    text: str
    start_line: int
    e

### 4.4 Embeddings (`embeddings.py`)

The embedder, with retries only for transient errors (rate limit, connection, timeout, 5xx) and exponential backoff with jitter.

In [11]:
show_source("docqa.embeddings")

# docqa.embeddings  (87 lines)

"""Embedding providers behind a small interface, so tests can inject a fake."""

import logging
import time
from collections.abc import Callable, Sequence
from typing import Protocol

import openai
from openai import OpenAI

from docqa.errors import ConfigError, EmbeddingError
from docqa.retry import call_with_retry, is_transient_api_error

logger = logging.getLogger(__name__)


class Embedder(Protocol):
    """Anything that turns text into vectors.

    `name` identifies the model; the store refuses to mix vectors from different names.
    """

    name: str

    def embed_documents(self, texts: Sequence[str]) -> list[list[float]]:
        """Embed many texts, returning one vector per text in the same order."""
        ...

    def embed_query(self, text: str) -> list[float]:
        """Embed a single search query."""
        ...


class OpenAIEmbedder:
    """Embedder for any OpenAI-compatible endpoint (OpenAI, OpenRouter, ...).

    The client is inje

### 4.5 Vector store (`store.py`)

Chroma store with idempotent ingestion. A file is keyed by a hash of its bytes plus the chunk settings, and a changed file is re-embedded before its old chunks are deleted.

In [12]:
show_source("docqa.store")

# docqa.store  (171 lines)

"""Persistent vector store with idempotent, hash-based ingestion."""

import hashlib
import logging
from collections.abc import Sequence
from dataclasses import dataclass
from pathlib import Path

import chromadb

from docqa.chunking import Chunk
from docqa.embeddings import Embedder
from docqa.errors import DocumentReadError, StoreError

logger = logging.getLogger(__name__)

COLLECTION_NAME = "docqa"
UPSERT_BATCH = 256
HASH_BLOCK_BYTES = 1024 * 1024


@dataclass(frozen=True)
class SearchHit:
    """One retrieved chunk with its citation fields and cosine distance (0 = identical)."""

    text: str
    source: str
    distance: float
    page: int | None = None
    section: str | None = None
    start_line: int | None = None
    end_line: int | None = None


def file_sha256(path: Path) -> str:
    """Hash a file's bytes, reading in blocks so large files stay cheap on memory."""
    digest = hashlib.sha256()
    try:
        with path.open("rb") as handle:
   

### 4.6 Retrieval (`retriever.py`)

Top-k search with a relevance threshold (1 minus cosine distance).

In [13]:
show_source("docqa.retriever")

# docqa.retriever  (48 lines)

"""Retrieve the chunks most relevant to a question, with a relevance guard."""

import logging
from dataclasses import dataclass

from docqa.errors import ConfigError
from docqa.store import SearchHit, VectorStore

logger = logging.getLogger(__name__)


@dataclass(frozen=True)
class RetrievedChunk:
    """A search hit plus its relevance score (1 - cosine distance; higher is better)."""

    hit: SearchHit
    relevance: float


class Retriever:
    """Top-k search over a vector store that drops weakly related chunks.

    A `relevance_threshold` of 0.0 keeps every hit (guard off). An empty result is
    not an error: it means nothing in the documents answers the question, and the
    answer step turns it into a fixed refusal message.
    """

    def __init__(self, store: VectorStore, top_k: int, relevance_threshold: float) -> None:
        if top_k < 1:
            raise ConfigError(f"top_k must be at least 1, got {top_k}")
        if not 0.0 <= relevanc

### 4.7 Answer generation (`answer.py`)

Builds the grounded prompt, calls the LLM, parses and normalises citation markers, and returns the fixed refusal when the answer is not in the documents.

In [14]:
show_source("docqa.answer")

# docqa.answer  (380 lines)

"""Grounded answer generation: prompt building, LLM call with retry, citations."""

import logging
import re
import time
from collections.abc import Callable, Iterable, Iterator, Sequence
from dataclasses import dataclass
from pathlib import Path
from typing import Any

import openai
from openai import OpenAI

from docqa.config import PROJECT_ROOT
from docqa.errors import ConfigError, InvalidQuestionError, LLMCallError
from docqa.retriever import RetrievedChunk
from docqa.retry import call_with_retry, is_transient_api_error

logger = logging.getLogger(__name__)

PROMPT_PATH = PROJECT_ROOT / "prompts" / "grounding.txt"
REFUSAL_MESSAGE = "I could not find this in the loaded documents."
SNIPPET_CHARS = 200
CITATION_RE = re.compile(r"\[(\d+)\]")
# Models sometimes write fullwidth brackets (【1】) or lists ([1, 2]); rewrite to [1][2].
MARKER_RE = re.compile(r"[\[【［]\s*(\d+(?:\s*[,，]\s*\d+)*)\s*[\]】］]")


@dataclass(frozen=True)
class Citation:
    """Where an answ

### 4.8 Pipeline (`pipeline.py`)

Ties ingest and ask together. Every external part is injected, so tests use fakes.

In [15]:
show_source("docqa.pipeline")

# docqa.pipeline  (237 lines)

"""Orchestration: ingest a folder of documents and answer questions about them."""

import logging
import time
from collections import defaultdict
from collections.abc import Callable, Iterator, Sequence
from dataclasses import dataclass
from pathlib import Path

from openai import OpenAI

from docqa.answer import (
    Answer,
    AnswerGenerator,
    AnswerStream,
    Turn,
    load_system_prompt,
    validate_question,
)
from docqa.chunking import Chunk, chunk_documents
from docqa.config import Settings
from docqa.conversation import ConversationMemory
from docqa.embeddings import OpenAIEmbedder
from docqa.errors import DocQAError
from docqa.loader import load_directory
from docqa.retriever import RetrievedChunk, Retriever
from docqa.store import VectorStore, content_key
from docqa.telemetry import QueryRecord, TelemetryLog, telemetry_path, utc_timestamp

logger = logging.getLogger(__name__)

BYTES_PER_MB = 1024 * 1024
TIMING_DECIMALS = 4


@dataclass(

### 4.9 Conversation memory (`conversation.py`)

Keeps the last N exchanges and rewrites a follow-up question into a standalone one before retrieval.

In [16]:
show_source("docqa.conversation")

# docqa.conversation  (38 lines)

"""Short-term conversation memory: the last few question and answer pairs."""

from collections import deque

from docqa.answer import Turn

DEFAULT_MAX_TURNS = 6


class ConversationMemory:
    """Remembers the most recent exchanges of one chat session; older ones are dropped."""

    def __init__(self, max_turns: int = DEFAULT_MAX_TURNS) -> None:
        if max_turns < 1:
            raise ValueError(f"max_turns must be at least 1, got {max_turns}")
        self._max_turns = max_turns
        self._turns: deque[Turn] = deque(maxlen=max_turns)

    @property
    def max_turns(self) -> int:
        """How many exchanges are remembered."""
        return self._max_turns

    @property
    def turns(self) -> tuple[Turn, ...]:
        """A snapshot of the remembered turns, oldest first."""
        return tuple(self._turns)

    def add(self, question: str, answer: str) -> None:
        """Remember one exchange; the oldest is dropped when the memory is ful

### 4.10 Telemetry (`telemetry.py`)

Appends one JSON line per question with timings and token counts, never the question text, and builds the `stats` summary.

In [17]:
show_source("docqa.telemetry")

# docqa.telemetry  (125 lines)

"""Per-question telemetry: stage timings and token counts, one JSON line per question."""

import json
import logging
import math
from collections.abc import Iterable
from dataclasses import asdict, dataclass
from datetime import UTC, datetime
from pathlib import Path

from docqa.config import Settings

logger = logging.getLogger(__name__)

TELEMETRY_FILENAME = "telemetry.jsonl"
P95 = 0.95


@dataclass(frozen=True)
class QueryRecord:
    """What one `ask` cost. The question text itself is never stored."""

    timestamp: str
    question_chars: int
    retrieved: int
    refused: bool
    retrieve_seconds: float
    generate_seconds: float
    prompt_tokens: int
    completion_tokens: int

    @property
    def total_seconds(self) -> float:
        """Retrieval plus generation time."""
        return self.retrieve_seconds + self.generate_seconds


@dataclass(frozen=True)
class Summary:
    """Aggregates over a set of records."""

    questions: int
    r

### 4.11 Retrieval metrics (`retrieval_metrics.py`)

Recall, precision, MRR and nDCG for the retrieval evaluation.

In [18]:
show_source("docqa.retrieval_metrics")

# docqa.retrieval_metrics  (77 lines)

"""Retrieval metrics with keyword-based relevance.

The question file has no relevance labels, so a chunk's relevance is the share of the
question's keywords found in what a reader sees in a citation: file name, section path
and chunk text. These are indicators for comparing changes, not ground truth: a chunk
can contain a keyword without answering the question.
"""

import math
import unicodedata
from collections.abc import Sequence
from pathlib import Path

from docqa.store import SearchHit

# A chunk counts as relevant when it holds at least this share of the keywords.
RELEVANT_SHARE = 0.5


def _normalize(text: str) -> str:
    """NFKC plus case folding, so ligatures and capitals do not hide a match."""
    return unicodedata.normalize("NFKC", text).casefold()


def hit_text(hit: SearchHit) -> str:
    """The text a reader sees in a citation: file name, section path and the chunk."""
    return " ".join(part for part in (Path(hit.source).stem,

### 4.12 Retrieval evaluation (`evaluate.py`)

Runs the evaluation questions through retrieval and reports the metrics for several `k` values.

In [19]:
show_source("docqa.evaluate")

# docqa.evaluate  (284 lines)

"""Retrieval evaluation: score how well the store ranks relevant chunks per question.

Relevance comes from each question's keywords (see docqa.retrieval_metrics).
Run with: ``uv run python -m docqa.evaluate``.
"""

import argparse
import json
import logging
import statistics
import sys
from collections import defaultdict
from collections.abc import Callable, Sequence
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Any

from dotenv import find_dotenv, load_dotenv
from openai import OpenAI

from docqa.config import PROJECT_ROOT, load_settings
from docqa.embeddings import OpenAIEmbedder
from docqa.errors import DocQAError, EvaluationError
from docqa.logging_setup import configure_logging
from docqa.retrieval_metrics import (
    hit_text,
    keyword_recall,
    keyword_share,
    ndcg_at_k,
    precision_at_k,
    reciprocal_rank,
)
from docqa.store import SearchHit, VectorStore

logger = logging.getLogger(__name__)

D

### 4.13 Answer evaluation (`evaluate_answers.py`)

Runs full question answering on the evaluation set (real API calls) and scores keyword coverage, refusals and citations.

In [20]:
show_source("docqa.evaluate_answers")

# docqa.evaluate_answers  (415 lines)

"""Answer-level evaluation: run questions through the whole pipeline and score the answers.

Compares retrieval depths (top_k) by the share of a question's keywords found in the answer,
and checks that unanswerable questions are refused. Keyword scoring is an indicator, not
ground truth: a paraphrase or a different number format can miss a keyword.
Run with: ``uv run python -m docqa.evaluate_answers``. It makes real API calls.
"""

import argparse
import json
import logging
import statistics
import sys
import time
from collections import defaultdict
from collections.abc import Callable, Mapping, Sequence
from dataclasses import asdict, dataclass
from pathlib import Path

from dotenv import find_dotenv, load_dotenv

from docqa.answer import Answer
from docqa.config import PROJECT_ROOT, Settings, load_settings
from docqa.errors import DocQAError, EvaluationError
from docqa.evaluate import DEFAULT_QUESTIONS, OVERALL, Question, load_questions
from doc

### 4.14 Command line (`cli.py`)

The commands `ingest`, `ask`, `chat` and `stats`, with exit codes: 0 success (a refusal is a success), 1 any error, 2 usage error.

In [21]:
show_source("docqa.cli")

# docqa.cli  (188 lines)

"""Command-line interface: ingest, ask, chat, and usage stats."""

import argparse
import json
import sys
from collections.abc import Callable, Sequence
from dataclasses import asdict
from pathlib import Path
from typing import Protocol

from dotenv import find_dotenv, load_dotenv

from docqa.answer import Answer
from docqa.config import Settings, load_settings
from docqa.conversation import DEFAULT_MAX_TURNS, ConversationMemory
from docqa.errors import DocQAError
from docqa.logging_setup import configure_logging
from docqa.pipeline import IngestSummary, build_pipeline
from docqa.telemetry import Summary, TelemetryLog, summarize, telemetry_path

EXIT_OK = 0
EXIT_FAILURE = 1  # argparse itself exits with 2 on a usage error
PROMPT = "> "
CLEAR_COMMAND = "/clear"
EXIT_COMMAND = "/exit"


class Pipeline(Protocol):
    """What the CLI needs from a pipeline."""

    def ingest(self, source_dir: Path) -> IngestSummary:
        """Embed every supported file in a folde

## 5. The grounding prompt

The system prompt lives in a file, not inside a Python string.

In [22]:
prompt_path = PROJECT / "docqa" / "prompts" / "grounding.txt"
if prompt_path.exists():
    print(prompt_path.read_text(encoding="utf-8"))
else:
    print("Prompt file not found at docqa/prompts/grounding.txt")

Prompt file not found at docqa/prompts/grounding.txt


## 6. The document corpus

The bundled corpus is 76 synthetic Markdown files about a fictional insurance-tech company, "Insurellm". PDF and plain-text files are supported too.

In [23]:
source_dir = PROJECT / "data" / "source_md"
files = sorted(source_dir.rglob("*.md")) if source_dir.exists() else []
print("Markdown files in data/source_md:", len(files))
for path in files[:8]:
    print("  ", path.relative_to(source_dir))

sample = next((p for p in files if p.name == "about.md"), files[0] if files else None)
if sample:
    print("\n--- first lines of", sample.name, "---")
    print("\n".join(sample.read_text(encoding="utf-8").splitlines()[:12]))

Markdown files in data/source_md: 76
   company/about.md
   company/careers.md
   company/culture.md
   company/overview.md
   contracts/Contract with Advantage Medical Coverage for Healthllm.md
   contracts/Contract with Apex Reinsurance for Rellm - AI-Powered Enterprise Reinsurance Solution.md
   contracts/Contract with Atlantic Risk Solutions for Bizllm.md
   contracts/Contract with Belvedere Insurance for Markellm.md

--- first lines of about.md ---
# About Insurellm

Insurellm was founded by Avery Lancaster in 2015 as an insurance tech startup designed to disrupt an industry in need of innovative products. Its first product was Markellm, the marketplace connecting consumers with insurance providers.

The company experienced rapid growth in its first five years, expanding its product portfolio to include Carllm (auto insurance portal), Homellm (home insurance portal), and Rellm (enterprise reinsurance platform). By 2020, Insurellm had reached a peak of 200 employees with 12 offices

## 7. Live run: ingest the documents

Ingestion loads each file, splits it into chunks, embeds them and stores them in Chroma. It is **idempotent**: a second run skips files that have not changed. The first run makes embedding API calls for all 76 files. It needs a valid API key in `.env`.

In [24]:
def run_docqa(*args: str, timeout: int = 900) -> subprocess.CompletedProcess:
    return subprocess.run(
        [sys.executable, "-m", "docqa", *args],
        capture_output=True, text=True, cwd=PROJECT, timeout=timeout,
    )


result = run_docqa("ingest")
print("exit code:", result.returncode)
print(result.stdout[-1500:])
print(result.stderr[-1000:])

exit code: 0
Ingest complete: 76 embedded, 0 unchanged, 0 failed; 1264 chunks in the store.

2026-11-28 15:15:50,917 INFO docqa.loader: loaded 76 documents from 76 files, skipped 0
2026-11-28 15:15:50,930 INFO docqa.chunking: split 76 documents into 632 chunks
2026-11-28 15:16:43,912 INFO docqa.pipeline: ingest finished: 76 embedded, 0 unchanged, 0 failed



Expected output on the first run, then on a second run that changes nothing:

```
Ingest complete: 76 embedded, 0 unchanged, 0 failed; 632 chunks in the store.
Ingest complete: 0 embedded, 76 unchanged, 0 failed; 632 chunks in the store.
```

In [25]:
# Run ingest again to show that it is idempotent: nothing should be re-embedded.
result = run_docqa("ingest")
print("exit code:", result.returncode)
print(result.stdout[-800:])

exit code: 0
Ingest complete: 0 embedded, 76 unchanged, 0 failed; 1264 chunks in the store.



## 8. Live run: ask a question and read the citations

The answer comes only from the retrieved passages. Each citation shows the file, the section and the line span.

In [26]:
result = run_docqa("ask", "Who founded Insurellm?")
print("exit code:", result.returncode)
print(result.stdout)

exit code: 0
Insurellm was founded by Avery Lancaster. [1]

Sources:
[1] about.md | About Insurellm, lines 1-5
    "# About Insurellm Insurellm was founded by Avery Lancaster in 2015 as an insurance tech startup designed to disrupt an industry in need of innovative products. Its first product was Markellm, the mark..."



The `--json` flag returns the same answer as structured data: the text, a list of citations, the refused flag and token counts.

In [27]:
result = run_docqa("ask", "--json", "When was Insurellm founded?")
print("exit code:", result.returncode)
try:
    answer = json.loads(result.stdout)
except json.JSONDecodeError:
    print(result.stdout[-1500:])
    print(result.stderr[-800:])
else:
    print("Answer:", answer.get("text"))
    print("Refused:", answer.get("refused"))
    print("Tokens:", answer.get("prompt_tokens"), "prompt +", answer.get("completion_tokens"), "completion")
    for citation in answer.get("citations", []):
        print(f"  [{citation.get('number')}] {citation.get('source')} | {citation.get('section')} "
              f"(lines {citation.get('start_line')}-{citation.get('end_line')})")

exit code: 0
Answer: Insurellm was founded in 2015. [1][3]
Refused: False
Tokens: 1468 prompt + 141 completion
  [1] overview.md | Overview of Insurellm (lines 1-5)
  [3] about.md | About Insurellm (lines 1-5)


## 9. Live run: a question the documents cannot answer

For an unrelated question, no retrieved chunk passes the relevance threshold, so the refusal is returned without calling the LLM. A refusal is still a success, so the exit code is `0`.

In [28]:
result = run_docqa("ask", "--json", "What is the capital of France?")
print("exit code:", result.returncode)
try:
    answer = json.loads(result.stdout)
except json.JSONDecodeError:
    print(result.stdout[-1000:])
else:
    print("Answer:", answer.get("text"))
    print("Refused:", answer.get("refused"))
    print("Citations:", answer.get("citations"))
    print("Tokens:", answer.get("prompt_tokens"), "prompt +", answer.get("completion_tokens"), "completion")

exit code: 0
Answer: I could not find this in the loaded documents.
Refused: True
Citations: []
Tokens: None prompt + None completion


## 10. Conversation memory

`chat` is an interactive session, so it is run in a terminal rather than in this notebook:

```bash
uv run python -m docqa chat
```

It remembers the last 6 question-and-answer exchanges (`--turns N` changes that), so a follow-up can use a pronoun:

```
> Who won the IIOTY award in 2023?
> What is her job title?
```

Each follow-up is first rewritten by the LLM into a standalone question, which is what retrieval searches with. The model then sees the earlier exchanges when it answers. Facts must still come from the numbered context passages, and citation numbers from old answers are stripped from the remembered text so they cannot be confused with the new ones. Type `/clear` to forget the history and `/exit` to quit. The code for this is in `docqa/conversation.py`, shown in section 4.9.

## 11. Telemetry

In [29]:
log = PROJECT / "data" / "telemetry.jsonl"
rows = []
if log.exists():
    rows = [json.loads(line) for line in log.read_text(encoding="utf-8").splitlines() if line.strip()]

if rows:
    refused = sum(1 for r in rows if r.get("refused"))
    prompt = sum(r.get("prompt_tokens", 0) for r in rows)
    completion = sum(r.get("completion_tokens", 0) for r in rows)
    print(f"Questions logged: {len(rows)} ({refused} refused)")
    print(f"Tokens: {prompt:,} prompt + {completion:,} completion")
    print("Last entry (the question text is never stored):")
    print(json.dumps(rows[-1], indent=2))
else:
    print("No telemetry log yet. Ask a question first.")

Questions logged: 13 (4 refused)
Tokens: 16,035 prompt + 1,362 completion
Last entry (the question text is never stored):
{
  "timestamp": "2026-11-28T09:46:55+00:00",
  "question_chars": 30,
  "retrieved": 0,
  "refused": true,
  "retrieve_seconds": 0.8207,
  "generate_seconds": 0.0001,
  "prompt_tokens": 0,
  "completion_tokens": 0
}


In [30]:
result = run_docqa("stats")
print("exit code:", result.returncode)
print(result.stdout)

exit code: 0
Questions: 13 (4 refused)
Latency (s): retrieve 1.30 | generate 1.64 | total 2.95 (p95 4.48)
Tokens: 16,035 prompt, 1,362 completion



## 12. Evaluation results

The evaluation set has 150 answerable questions in 7 categories, plus 5 unrelated and 6 near-topic questions that must be refused. Relevance and answer correctness are **keyword proxies**: a chunk or an answer counts as covering a question when it contains the question's keywords. The numbers compare settings with each other. They are not ground truth.

### Retrieval

| k | recall | precision | MRR | nDCG | spanning recall |
|---|---|---|---|---|---|
| 4 | 0.929 | 0.773 | 0.933 | 0.945 | 0.756 |
| 6 | 0.948 | 0.703 | 0.936 | 0.931 | 0.808 |
| 8 | 0.957 | 0.643 | 0.937 | 0.923 | 0.853 |
| 10 | 0.963 | 0.584 | 0.938 | 0.921 | 0.898 |

The file name counts as chunk text in this evaluation, so these are upper bounds.

### Full answers (real API calls)

| top_k | keyword coverage | all keywords | answerable but refused | cited | prompt tokens | seconds |
|---|---|---|---|---|---|---|
| 6 | 0.707 | 0.527 | 6.7% | 100% | 1,050 | 2.36 |
| 10 | 0.755 | 0.620 | 5.3% | 100% | 1,586 | 2.57 |

| category | k=6 | k=10 |
|---|---|---|
| comparative | 0.900 | 1.000 |
| direct_fact | 0.740 | 0.793 |
| holistic | 0.500 | 0.550 |
| numerical | 0.800 | 0.633 |
| relationship | 0.900 | 0.900 |
| spanning | 0.363 | 0.422 |
| temporal | 0.800 | 0.925 |

`top_k` 10 is the default. It improved coverage by 0.048 for 51% more prompt tokens.

**Refusals:** all 11 unanswerable questions were refused at both settings. The 5 unrelated ones were stopped by the relevance threshold with no LLM call. The 6 near-topic ones passed the threshold and were refused by the model, so for those the prompt is the only protection.

In [31]:
# Optional: re-run the evaluation yourself. It may call the embedding API, and the answer
# evaluation makes many real LLM calls, so both are switched off by default.
RUN_RETRIEVAL_EVALUATION = False
RUN_ANSWER_EVALUATION = False

if RUN_RETRIEVAL_EVALUATION:
    result = subprocess.run([sys.executable, "-m", "docqa.evaluate"], capture_output=True, text=True, cwd=PROJECT)
    print(result.stdout[-3000:])
    print(result.stderr[-800:])

if RUN_ANSWER_EVALUATION:
    result = subprocess.run([sys.executable, "-m", "docqa.evaluate_answers"], capture_output=True, text=True, cwd=PROJECT)
    print(result.stdout[-3000:])
    print(result.stderr[-800:])

eval_dir = PROJECT / "evaluation"
if eval_dir.exists():
    print("Files in evaluation/:")
    for path in sorted(eval_dir.iterdir()):
        print("  ", path.name)

Files in evaluation/:
   README.md
   near_topic.jsonl
   results
   tests.jsonl
   unanswerable.jsonl


## 13. Tests

In [32]:
tests = subprocess.run(
    [sys.executable, "-m", "pytest", "-q"],
    capture_output=True, text=True, cwd=PROJECT,
)
print(tests.stdout[-1500:])
print(tests.stderr[-500:])

................................................ [ 71%]
........................................................................ [ 95%]
..............                                                           [100%]
302 passed in 4.87s




Every external part (embedder, LLM client, store) is injected, so the tests use fakes and need no network and no API key. Nothing runs at import time.

## 14. Design decisions, limitations, skills

**Design decisions**

- **LangChain only where it saves work.** `RecursiveCharacterTextSplitter` does the splitting. Chroma and the OpenAI-compatible client are used directly, so retries, citations and the refusal path stay explicit and testable.
- **Heading-aware chunks with line spans**, and the file name in the embedded header, so citations point to a section and lines, and a question that names a file or person matches it.
- **Idempotent ingestion.** Files are keyed by a content hash. A changed file is re-embedded first and its old chunks deleted second, so a failed API call keeps the previous version.
- **Per-file isolation.** One unreadable file or failed embedding is reported and the rest continue. The exit code is still `1`.
- **Retries only for transient errors**, with exponential backoff and jitter. A bad API key or a bad request fails immediately.
- **Citation markers are parsed and normalised.** The model sometimes writes `【1】` or `[1, 2]`. Both become ordinary `[n]` references, and only cited passages are listed as sources.
- **Typed errors.** Every failure the program expects is a `DocQAError` subclass, so the CLI turns it into a one-line message and exit code `1`.

**Limitations**

- **Aggregation questions** ("total value of all Healthllm contracts") need every matching document. Top-k retrieval cannot gather them, so the assistant refuses. Refusing is safer than a wrong total.
- **Hidden hops** such as "the salary of the founder of Insurellm" need two lookups, and some are refused.
- **Spanning questions** are the weakest category (coverage 0.42).
- **Numerical questions got worse at k=10** (0.800 to 0.633). This is a guess from a single run.
- **Near-topic refusals rely on the model.** Only 6 such questions were tested.
- PDF text comes from plain text extraction, so columns and tables can come out jumbled, and scanned PDFs without a text layer have nothing to read.

**Skills shown in this project:** Python, Retrieval-Augmented Generation (RAG), vector embeddings, Chroma, semantic search, text chunking, prompt engineering, grounded generation, citations, LLM API integration, retry and exponential backoff, idempotent data pipelines, evaluation metrics (recall, precision, MRR, nDCG), telemetry and logging, CLI design, pytest, ruff, uv, dependency injection.